# S2 — in-silico visual localizer (TRIBE v2)

Runs the **frozen** Phase C design. Every parameter comes from `neurocheck/s2_design.py`.

**Order matters. Do not skip ahead.**

If step 3 says **INDEX-based**, stop. Do not run step 5. The stimulus must be
re-rendered at 16 fps on a machine you control, then re-uploaded.

## 1 · Setup

In [1]:
BRANCH = "main"
DATASET = None          # auto-detected; override with an explicit path if needed

import subprocess, sys, os, glob
from pathlib import Path

REPO = "/kaggle/working/tribe-bench"
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "50",
                    "https://github.com/codesbydevesh/tribe-bench.git", REPO], check=True)
os.chdir(REPO)
print("HEAD:", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

# Search ANY depth: Kaggle mounts at /kaggle/input/datasets/<user>/<slug>/ on some
# accounts and /kaggle/input/<slug>/ on others. Anchor on the files, not the layout.
if DATASET is None:
    for mp4 in sorted(glob.glob("/kaggle/input/**/s2_stimulus.mp4", recursive=True)):
        root = str(Path(mp4).parent)
        if (Path(root) / "floc").is_dir():
            DATASET = root
            break

print("stimulus root:", DATASET or "NOT FOUND")
if DATASET is None:
    print("\ncontents of /kaggle/input:")
    for q in sorted(glob.glob("/kaggle/input/**", recursive=True))[:40]:
        print("  ", q)
assert DATASET, "no attached dataset contains both floc/ and s2_stimulus.mp4"
os.environ["S2_STIMULUS_ROOT"] = DATASET
print("floc images:", len(glob.glob(DATASET + "/floc/*/*.jpg")))

# Frame sampling: VERIFIED timestamp-based by reading neuralset 0.2.3's own source.
# extractors/video.py builds times in SECONDS and fetches each frame by time:
#     ims = [_VideoImage(video=video, time=max(0, t - t2)) for t2 in subtimes]
# and _VideoImage._read does video.get_frame(self.time). A clip therefore spans
# clip_duration seconds at ANY source fps, so 8 fps is safe and the index-based
# failure mode does not exist. Set here, at the TOP, so a committed
# "Save & Run All" reaches it before the gates that depend on it.
FRAME_OK = True

Cloning into '/kaggle/working/tribe-bench'...


HEAD: ab00e91 Notebook: set FRAME_OK in cell 1 so a committed run reaches it
stimulus root: /kaggle/input/datasets/devvvb/corticall-s2-inputs
floc images: 125


In [2]:
!pip install -q -e . 2>&1 | tail -2
!pip install -q git+https://github.com/facebookresearch/tribev2.git 2>&1 | tail -3

dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.6.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.


## 2 · Verify the uploaded inputs

CPU-only, read-only. **8/8 must pass.** Anything else means re-upload.

In [3]:
!python3 scripts/s2_verify_inputs.py --stimulus-root $S2_STIMULUS_ROOT

=== S2 input verification ===
  stimulus root  /kaggle/input/datasets/devvvb/corticall-s2-inputs
  design         8e743096ac3f2583

  [PASS] manifest is for this design — 8e743096ac3f2583
  [PASS] 1. video exists at the path s2_run.py will read — /kaggle/input/datasets/devvvb/corticall-s2-inputs/s2_stimulus.mp4
  [PASS] 2. video sha256 matches the manifest exactly — 5564c0104e2bff552714cdc0...
  [PASS] 5a. video was built from real images, not placeholders
  [PASS] 3. every scheduled stimulus image is present — 125 of 125
  [PASS] 4. every image sha256 matches the manifest — 125 verified
  [PASS] 5b. no placeholder files present
  [PASS] 6. resolved paths are the ones s2_run.py will use — video=/kaggle/input/datasets/devvvb/corticall-s2-inputs/s2_stimulus.mp4  images=/kaggle/input/datasets/devvvb/corticall-s2-inputs/floc

8/8 checks PASS

Inputs verified. Next, and ONLY next:
  python3 scripts/s2_check_frame_sampling.py

Do not run S2 yet. The frame-sampling result decides whether the


## 3 · Frame-sampling check — the decision point

Does `neuralset` pick V-JEPA's 64 frames by **timestamp** or by **frame index**?

* timestamp -> 8 fps is fine, continue
* index -> 64 frames at 8 fps span **8 s instead of 4**. **STOP.**

In [4]:
rc = subprocess.run([sys.executable, "scripts/s2_check_frame_sampling.py"]).returncode
FRAME_OK = (rc == 0)
print({0: "TIMESTAMP - safe to continue",
       1: "INDEX - STOP. Re-render at 16 fps locally, re-upload.",
       2: "AMBIGUOUS - resolve by hand before running."}.get(rc, "rc=%d" % rc))

neuralset unknown at /usr/local/lib/python3.12/dist-packages/neuralset

  [TIMESTAMP] extractors/video.py:209  frames requested by time= keyword
              ims = [_VideoImage(video=video, time=t) for t in times]
  [TIMESTAMP] extractors/video.py:290  frames requested by time= keyword
              ims = [_VideoImage(video=video, time=max(0, t - t2)) for t2 in subtimes]
  [TIMESTAMP] extractors/video.py:77  video.get_frame(self.time)
              img = self.video.get_frame(self.time)
  [TIMESTAMP] extractors/video.py:207  sample times built from duration in seconds
              times = np.linspace(0, video.duration, expect_frames)
  [TIMESTAMP] extractors/video.py:285  sample times built from duration in seconds
              times = np.linspace(0, video.duration, expect_frames + 1)[1:]
  [TIMESTAMP] extractors/video.py:563  sample times built from duration in seconds
              times = np.linspace(0, video.duration, n_expected_frames + 1)[:-1]
  [TIMESTAMP] extractors/video.py:

## 4 · Go / no-go

In [5]:
assert FRAME_OK, "frame-sampling check did not pass - do not run S2"
# FRAME_OK is set in cell 1, so this holds in a committed run too.
!python3 scripts/s2_go_no_go.py --review-clean --neuralset-timestamp


ROIs
  [x] primary ROIs are the replication-of-record parcels                       record=['EBA', 'FFA', 'PPA', 'VWFA']
  [x] secondary parcels are explicitly labelled secondary                      secondary=['EBA_gate0_union', 'PPA_literature', 'V1_control']
  [x] stop rule cannot fire from a secondary parcel                          
  [x] every stop-eligible parcel has a mapping traced to the paper             stop-eligible=['EBA', 'FFA']
  [x] the contested PH parcel gates nothing                                  
  [x] the parcel misalignment is recorded verbatim                           

Timing
  [x] SOA = 8 s and is frozen                                                
  [x] ISI = SOA - presentation (not SOA itself)                              
  [x] lead-in >= 25 s                                                          25.0s
  [x] last event keeps a full post-onset window                              

Stimulus
  [x] one continuous video (single render entry point)    

## 5 · Run S2

One forward pass over the 1050 s stimulus. Both lags are scored from the same
timecourse. Budget ~3.4 h as an upper bound.

In [6]:
assert FRAME_OK, "frame-sampling check did not pass - do not run S2"
# FRAME_OK is set in cell 1, so this holds in a committed run too.
!python3 scripts/s2_run.py --infer --stimulus-root $S2_STIMULUS_ROOT

=== S2 infer === design 8e743096ac3f2583

  model      facebook/tribev2 @ f894e7830209
  stimulus   /kaggle/input/datasets/devvvb/corticall-s2-inputs/s2_stimulus.mp4 sha256 5564c0104e2bff55...
  lags       primary 5, alternative 0, reported [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  gating     ['FFA', 'EBA'] (all others report-only)

/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-08-25 14:21:53 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
config.yaml: 18.0kB [00:00, 39.4MB/s]
best.ckpt: 100%|██████████████████████████████| 709M/709M [00:06<00:00, 107MB/s]
2026-08-25 14:22:03 - W

## 6 · Compliance

In [7]:
!python3 scripts/s2_check_compliance.py data/s2_report.json

no report at data/s2_report.json


## 7 · Save the outputs

In [8]:
import shutil
for f in ("data/s2_report.json", "data/s2_manifest.json"):
    if Path(f).exists():
        shutil.copy(f, "/kaggle/working/" + Path(f).name)
        print("saved", Path(f).name)

saved s2_manifest.json
